# Word2Vec: CBOW vs Skip-Gram

Word2Vec trains a neural network to predict words from context, producing dense vector representations where semantically similar words cluster together. Two architectures exist: CBOW predicts a target word from surrounding context words, while Skip-Gram does the reverse. Both produce word embeddings but differ in training dynamics and performance characteristics — CBOW is faster and suits frequent words; Skip-Gram handles rare words better and generally produces richer representations on large corpora.


## Install and Import

In [ ]:
!pip install gensim -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.3 MB/s eta 0:00:00
Ready.


## Text Dataset

A small corpus covering animals, royalty, geography and nature. In production, replace this with Wikipedia, news archives, or domain-specific text — larger and more varied corpora produce significantly better embeddings.


In [3]:
import re
from gensim.models import Word2Vec

print("Ready.")


Ready.


In [6]:
raw_text = """
Ultralytics YOLO26 is a versatile AI framework that supports multiple computer visiontasks. The framework can be used to perform detection, segmentation, OBB, classification, and pose estimation. Each of these tasks has a different objective and use case, allowing you to address various computer vision challenges with a single framework.
Detection is the primary task supported by YOLO26. It involves identifying objects in an image or video frame and drawing bounding boxes around them. The detected objects are classified into different categories based on their features. YOLO26 can detect multiple objects in a single image or video frame with high accuracy and speed, making it ideal for real-time applications like surveillance systems and autonomous vehicles.
Segmentation takes object detection further by producing pixel-level masks for each object. This precision is useful for applications such as medical imaging, agricultural analysis, and manufacturing quality control.
Classification involves categorizing entire images based on their content. This task is essential for applications like product categorization in e-commerce, content moderation, and wildlife monitoring.
Pose estimation detects specific keypoints in images or video frames to track movements or estimate poses. These keypoints can represent human joints, facial features, or other significant points of interest. YOLO26 excels at keypoint detection with high accuracy and speed, making it valuable for fitness applications, sports analytics, and human-computer interaction.
Oriented Bounding Box (OBB) detection enhances traditional object detection by adding an orientation angle to better locate rotated objects. This capability is particularly valuable for aerial imagery analysis, document processing, and industrial applications where objects appear at various angles. YOLO26 delivers high accuracy and speed for detecting rotated objects in diverse scenarios.
Ultralytics YOLO26 supports multiple computer vision tasks, including detection, segmentation, classification, oriented object detection, and keypoint detection. Each task addresses specific needs in the computer vision landscape, from basic object identification to detailed pose analysis. By understanding the capabilities and applications of each task, you can select the most appropriate approach for your specific computer vision challenges and leverage YOLO26's powerful features to build effective solutions.
"""

print(f"Corpus loaded.")


Corpus loaded.


## Preprocessing

Lowercase the text, strip punctuation, and split into sentences. Each sentence becomes a list of tokens — the format gensim expects.


In [7]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    sentences = []
    for line in text.strip().split('\n'):
        words = line.strip().split()
        if len(words) > 1:
            sentences.append(words)
    return sentences

sentences = preprocess(raw_text)
all_words = [w for s in sentences for w in s]

print(f"Sentences : {len(sentences)}")
print(f"Tokens    : {len(all_words)}")
print(f"Unique    : {len(set(all_words))}")
print(f"\nSample:")
for s in sentences[:3]:
    print(f"  {' '.join(s)}")


Sentences : 7
Tokens    : 336
Unique    : 183

Sample:
  ultralytics yolo is a versatile ai framework that supports multiple computer visiontasks the framework can be used to perform detection segmentation obb classification and pose estimation each of these tasks has a different objective and use case allowing you to address various computer vision challenges with a single framework
  detection is the primary task supported by yolo it involves identifying objects in an image or video frame and drawing bounding boxes around them the detected objects are classified into different categories based on their features yolo can detect multiple objects in a single image or video frame with high accuracy and speed making it ideal for realtime applications like surveillance systems and autonomous vehicles
  segmentation takes object detection further by producing pixellevel masks for each object this precision is useful for applications such as medical imaging agricultural analysis and manufact

## Train Both Models

Key parameters: `vector_size` sets embedding dimensions; `window` controls context span; `min_count` ignores rare tokens; `sg=0` selects CBOW, `sg=1` selects Skip-Gram; `epochs` controls training passes.


In [8]:
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=0,        # CBOW
    epochs=200,
    seed=42
)

sg_model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,        # Skip-Gram
    epochs=200,
    seed=42
)

print("Both models trained.")


Both models trained.


## Vocabulary Size

Both models share the same vocabulary since they train on the same corpus. Every unique token gets a 50-dimensional vector.


In [9]:
vocab = list(cbow_model.wv.key_to_index.keys())

print(f"Vocabulary size: {len(vocab)} words")
print(f"\nAll tokens:")
print(sorted(vocab))


Vocabulary size: 183 words

All tokens:
['a', 'accuracy', 'adding', 'address', 'addresses', 'aerial', 'agricultural', 'ai', 'allowing', 'an', 'analysis', 'analytics', 'and', 'angle', 'angles', 'appear', 'applications', 'approach', 'appropriate', 'are', 'around', 'as', 'at', 'autonomous', 'based', 'basic', 'be', 'better', 'bounding', 'box', 'boxes', 'build', 'by', 'can', 'capabilities', 'capability', 'case', 'categories', 'categorization', 'categorizing', 'challenges', 'classification', 'classified', 'computer', 'content', 'control', 'delivers', 'detailed', 'detect', 'detected', 'detecting', 'detection', 'detects', 'different', 'diverse', 'document', 'drawing', 'each', 'ecommerce', 'effective', 'enhances', 'entire', 'essential', 'estimate', 'estimation', 'excels', 'facial', 'features', 'fitness', 'for', 'frame', 'frames', 'framework', 'from', 'further', 'has', 'high', 'human', 'humancomputer', 'ideal', 'identification', 'identifying', 'image', 'imagery', 'images', 'imaging', 'in', 'incl

## Word Vectors

Each word is encoded as a list of 50 floats. The absolute values are not interpretable individually — what matters is the geometric relationship between vectors. Only the first 10 values are shown here.


In [12]:
a = ['ultralytics', 'yolo', 'detection', 'image', 'classification']

print(f"{'Word':<10}  {'Model':<10}  Vector (first 10 values)")
print("-" * 72)

for b in a:
    for label, model in [("CBOW", cbow_model), ("Skip-Gram", sg_model)]:
        vec = model.wv[b]
        vals = ', '.join([f'{v:.3f}' for v in vec[:10]])
        print(f"{b:<10}  {label:<10}  [{vals} ...]")
    print()


Word        Model       Vector (first 10 values)
------------------------------------------------------------------------
ultralytics  CBOW        [0.001, -0.135, -0.212, 0.103, 0.041, 0.070, -0.007, 0.273, -0.336, 0.026 ...]
ultralytics  Skip-Gram   [-0.199, -0.385, -0.345, 0.190, 0.201, 0.070, 0.298, 0.264, -0.338, -0.108 ...]

yolo        CBOW        [0.041, -0.224, -0.423, 0.211, 0.101, 0.089, -0.098, 0.481, -0.693, -0.000 ...]
yolo        Skip-Gram   [0.088, -0.468, -0.469, 0.380, 0.146, -0.148, -0.012, 0.098, -0.406, -0.164 ...]

detection   CBOW        [0.025, -0.174, -0.307, 0.152, 0.081, 0.055, -0.072, 0.416, -0.628, -0.007 ...]
detection   Skip-Gram   [-0.029, 0.067, -0.139, 0.090, 0.019, -0.122, 0.244, 0.250, -0.810, -0.540 ...]

image       CBOW        [0.022, -0.160, -0.235, 0.098, 0.032, 0.040, -0.077, 0.317, -0.444, 0.003 ...]
image       Skip-Gram   [0.118, -0.333, -0.341, 0.294, 0.063, 0.091, -0.205, 0.299, -0.522, -0.168 ...]

classification  CBOW        [0.001, -0.11

## Top-5 Similar Words

Similarity is measured by cosine distance between vectors. The two models often agree on the top candidates but differ in ranking, reflecting their different training objectives.


In [11]:
new_vec = ['ultralytics', 'yolo', 'detection', 'image', 'classification']

for x in new_vec:
    print(f"Query: '{x}'")
    print(f"  {'Rank':<5}  {'CBOW':<28}  {'Skip-Gram':<28}")
    print(f"  {'----':<5}  {'-'*26:<28}  {'-'*26:<28}")

    cbow_sim = cbow_model.wv.most_similar(x, topn=5)
    sg_sim   = sg_model.wv.most_similar(x, topn=5)

    for i, ((cw, cs), (sw, ss)) in enumerate(zip(cbow_sim, sg_sim), 1):
        cb = f"{cw:<14} {cs:.4f}"
        sk = f"{sw:<14} {ss:.4f}"
        print(f"  {i:<5}  {cb:<28}  {sk:<28}")
    print()


Query: 'ultralytics'
  Rank   CBOW                          Skip-Gram                   
  ----   --------------------------    --------------------------  
  1      supports       0.9946         supports       0.9659       
  2      yolo           0.9942         versatile      0.9633       
  3      multiple       0.9942         ai             0.9604       
  4      ai             0.9936         that           0.9553       
  5      framework      0.9933         framework      0.9502       

Query: 'yolo'
  Rank   CBOW                          Skip-Gram                   
  ----   --------------------------    --------------------------  
  1      at             0.9965         excels         0.8692       
  2      various        0.9956         interest       0.8361       
  3      objects        0.9954         at             0.8319       
  4      involves       0.9953         angles         0.8189       
  5      angles         0.9951         single         0.7984       

Query: 'det

## Summary

**CBOW** averages context embeddings to predict a center word. It trains faster and performs well when the corpus is small or words are frequent.

**Skip-Gram** treats each context word as a separate training example. It takes longer to train but generalises better to rare words and typically produces higher-quality embeddings on large corpora.

Both models represent words as dense vectors in a shared geometric space. Distance and direction in that space encode semantic relationships — synonyms cluster, antonyms tend to oppose, and analogy tasks resolve to simple vector arithmetic.
